In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import h5py
from hydra import compose, initialize_config_dir
import numpy as np
from omegaconf import OmegaConf
import pandas as pd
from morphemes import Morphemes
from tqdm.auto import tqdm

from src.analysis.state_space import StateSpaceAnalysisSpec, \
    prepare_state_trajectory, aggregate_state_trajectory, flatten_trajectory
from src.analysis import analogy_pseudocausal
from src.datasets.speech_equivalence import SpeechHiddenStateDataset

In [3]:
base_model = "w2v2_pc_8"

model_class = "ffff_32-pc-phon_mAP1"#discrim-rnn_32-pc-mAP1"
model_name = "phoneme_10frames_fixedlen25"

train_dataset = "librispeech-train-clean-100"
dataset = train_dataset

# hidden_states_path = f"outputs/hidden_states/{base_model}/{train_dataset}.h5"
hidden_states_path = f"/scratch/jgauthier/{base_model}_{train_dataset}.h5"
embeddings_path = f"outputs/model_embeddings/{train_dataset}/{base_model}/{model_class}/{model_name}/{dataset}.npy"

ss_path = f"outputs/state_space_specs/librispeech-train-clean-100/{base_model}/state_space_specs.h5"

base_model_class = base_model.rsplit("_", maxsplit=1)[0]
outdir = f"outputs/analogy_morph/inputs/{dataset}/{base_model_class}"

min_samples_per_word = 2
max_samples_per_word = 100

min_inflection_samples = 5

In [4]:
if embeddings_path == "ID":
    model_representations = SpeechHiddenStateDataset.from_hdf5(hidden_states_path).states
else:
    with open(embeddings_path, "rb") as f:
        model_representations: np.ndarray = np.load(f)
ss = StateSpaceAnalysisSpec.from_hdf5(ss_path, "word")
assert ss.is_compatible_with(model_representations)

In [5]:
ss = ss.subsample_instances(max_samples_per_word)

In [6]:
trajectory = prepare_state_trajectory(model_representations, ss)
trajectory = aggregate_state_trajectory(trajectory, ss, ("mean_within_cut", "phoneme"))

  0%|          | 0/32046 [00:00<?, ?it/s]

Aggregating:   0%|          | 0/32046 [00:00<?, ?label/s]

In [7]:
agg, agg_src = flatten_trajectory(trajectory)

In [8]:
flat_idx_lookup = {(label_idx, instance_idx, phoneme_idx): flat_idx
                    for flat_idx, (label_idx, instance_idx, phoneme_idx) in enumerate(agg_src)}

In [9]:
label_to_idx = {label: idx for idx, label in enumerate(ss.labels)}

In [10]:
cuts_df = ss.cuts.xs("phoneme", level="level").drop(columns=["onset_frame_idx", "offset_frame_idx"])
cuts_df["label_idx"] = cuts_df.index.get_level_values("label").map({l: i for i, l in enumerate(ss.labels)})
cuts_df["frame_idx"] = cuts_df.groupby(["label", "instance_idx"]).cumcount()
cuts_df = cuts_df.reset_index().set_index(["label", "instance_idx", "frame_idx"]).sort_index()

In [11]:
cut_phonemic_forms = cuts_df.groupby(["label", "instance_idx"]).description.agg(' '.join)

In [12]:
inflection_results = pd.read_parquet("outputs/analogy/inputs/librispeech-train-clean-100/w2v2_pc/inflection_results.parquet")

In [13]:
inflection_instances = pd.read_parquet("outputs/analogy/inputs/librispeech-train-clean-100/w2v2_pc/all_cross_instances.parquet")

## Overall store for results

In [14]:
all_cross_instances = []
expts = {}

In [15]:
def get_next_phoneme_cohort(xs, min_instances=min_inflection_samples):
    next_phoneme = xs.str.split(" ").str[0]
    cohort = next_phoneme.value_counts()
    cohort = cohort[cohort >= min_instances].index.tolist()
    if not cohort:
        return "|"
    return "|" + "|".join(sorted(cohort)) + "|"

## Prepare counterfactual NNS/VBZ inflections

In [16]:
study_inflections = ["VBZ", "NNS"]
study_post_divergences = ["Z", "S", "IH Z"]
exclude_nns_vbz_bases = "know seem please write meet read do".split()
study_inflection_instances = inflection_instances[
    inflection_instances.inflection.str.startswith(tuple(study_inflections))
    & inflection_instances.is_regular
    & (inflection_instances.base_ambig_NN_VB == False)
    & inflection_instances.post_divergence.isin(study_post_divergences)
    & ~inflection_instances.base.isin(exclude_nns_vbz_bases)
].copy()

In [17]:
study_inflection_instances["next_phoneme"] = study_inflection_instances.post_divergence.str.split(" ").str[0]
study_inflection_instances["next_phoneme_idx"] = study_inflection_instances.base_phones.str.count(" ") + 1
study_inflection_instances["next_phoneme_cohort"] = study_inflection_instances.groupby(["inflection", "base_phones"]) \
    .post_divergence.transform(get_next_phoneme_cohort)

In [18]:
# Only retain forms for which there are counterfactual post-divergence targets
# This works for prefixes only at the moment
broad_cf_instances = []

for (base_phones, post_divergence), rows in tqdm(study_inflection_instances.groupby(["base_phones", "post_divergence"])):
    # find phonemic forms which have this base with a *different* inflection
    matches = cut_phonemic_forms[cut_phonemic_forms.str.startswith(base_phones + " ")]
    matches = matches[~matches.str.endswith(" " + post_divergence)].to_frame()
    if matches.empty:
        continue

    matches["post_divergence"] = matches.description.str.replace(base_phones + " ", "")
    matches["inflected_idx"] = matches.index.get_level_values("label").map(label_to_idx)

    for row in matches.itertuples():
        label, instance_idx = row.Index
        broad_cf_instances.append((base_phones, post_divergence, label, instance_idx,
                                   row.inflected_idx, row.description, row.post_divergence))

broad_cf_instances = pd.DataFrame(
    broad_cf_instances,
    columns=["base_phones", "orig_post_divergence",
             "inflected", "inflected_instance_idx", "inflected_idx",
             "inflected_phones", "post_divergence"])

  0%|          | 0/909 [00:00<?, ?it/s]

In [19]:
# Merge in all the product of all possible base instances.
# To prepare for this, we'll un-product the instances df, just taking the base and dropping
# the inflected information.
inflected_base_instances = study_inflection_instances.groupby(
        ["inflection", "base_phones", "post_divergence",
        "base", "base_idx", "base_instance_idx"]) \
    .first().index.to_frame(index=False)

broad_cf_instances = broad_cf_instances.merge(
    inflected_base_instances,
    left_on=["base_phones", "orig_post_divergence"],
    right_on=["base_phones", "post_divergence"],
    how="left",
    suffixes=("", "_base")).drop(columns=["post_divergence_base"])

In [20]:
# NB this applies just for prefixes
broad_cf_instances["next_phoneme_idx"] = broad_cf_instances.base_phones.str.count(" ") + 1

broad_cf_instances["next_phoneme"] = broad_cf_instances.post_divergence.str.split(" ").str[0]
broad_cf_instances["base_inflection"] = broad_cf_instances.inflection
broad_cf_instances["inflection"] = broad_cf_instances.inflection + "-" + broad_cf_instances.next_phoneme
broad_cf_instances["counterfactual"] = False

In [21]:
broad_cf_instances["next_phoneme_cohort"] = broad_cf_instances.groupby(["base_phones"]).post_divergence.transform(get_next_phoneme_cohort)

In [22]:
# Now generate transfer targets.
# Consider a given cohort like "follow", which has the VBZ inflection along with counterfactual
# continuations /D/ 'followed', /IH NG/ 'following', /ER/ 'follower'.
# A transfer trial might ask that we take the /Z/ of a distinct VBZ type (e.g. "adds") and
# apply it to the base "follow" of a token of "followed", and try to reach /Z/ "follows".
#
# We are just generating the right side (target) of those trials here. We do that by taking
# the actual observed tokens of e.g. "followed" and asserting that the next phoneme should
# actually be /Z/ and so on.
#
# We are evaluating transfer both from VBZ/NNS to counterfactuals, as well as between
# counterfactuals. In practice the sounds generated by the VBZ/NNS are a subset of those
# present in possible counterfactual sources. We'll double check this and then just use
# the counterfactual data to generate transfer trials.

In [23]:
# gt_to_cf_trials = []

# gt_base_divergences = ["Z", "S", "IH Z"]  # TODO hard-coded
# assert set(gt_base_divergences) <= set(study_inflection_instances.post_divergence.unique())
# for div_content in gt_base_divergences:
#     # Find items that could potentially be used as transfer targets.
#     div_next_phoneme = div_content.split(" ")[0]
#     match = broad_cf_instances[broad_cf_instances.next_phoneme != div_next_phoneme]
#     match = match[match.next_phoneme_cohort.str.contains(f"|{div_next_phoneme}|", regex=False)]

#     match["post_divergence"] = div_content
#     match["next_phoneme"] = div_content.split(" ")[0]
#     match["base_inflection"] = match.inflection
#     match["inflection"] = match.inflection + f"-{div_content}"
#     match["inflected_phones"] = match.base_phones + " " + div_content.split(" ")[0]
#     match = match.drop(columns=["next_phoneme_cohort"])
    
#     gt_to_cf_trials.append(match)

# gt_to_cf_trials = pd.concat(gt_to_cf_trials, ignore_index=True)

In [24]:
cf_to_cf_trials = []

vbz_nns_next_phonemes = ["Z", "S", "IH"]  # TODO hard-coded
cf_next_phonemes = broad_cf_instances.next_phoneme.unique()
assert set(vbz_nns_next_phonemes) <= set(cf_next_phonemes)

for next_phoneme in tqdm(cf_next_phonemes):
    match = broad_cf_instances[broad_cf_instances.next_phoneme != next_phoneme]
    match = match[match.next_phoneme_cohort.str.contains(f"|{next_phoneme}|", regex=False)]
    match = match.copy()

    match["post_divergence"] = next_phoneme
    match["next_phoneme"] = next_phoneme
    match["base_inflection"] = match.base_inflection
    match["inflection"] = match.inflection.str.rsplit("-", n=1).str[0] + f"-ctf-{next_phoneme}"
    match["inflected_phones"] = match.base_phones + " " + next_phoneme
    match["counterfactual"] = True
    match = match.drop(columns=["next_phoneme_cohort"])

    cf_to_cf_trials.append(match)

cf_to_cf_trials = pd.concat(cf_to_cf_trials, ignore_index=True)

  0%|          | 0/38 [00:00<?, ?it/s]

In [25]:
all_cross_instances.append(broad_cf_instances)
all_cross_instances.append(cf_to_cf_trials)

In [26]:
for target_phoneme in tqdm(cf_next_phonemes):
    for source_inflection in study_inflections:
        for source_orig_phoneme in vbz_nns_next_phonemes:
            # if source_true:
            #     source_rows = inflection_instances.query(f"inflection == '{source_inflection}-{source_phoneme}'")
            #     source_next_phons = source_rows.post_divergence.unique()
            # else:
            #     source_rows = broad_cf_instances.query(f"inflection == '{source_inflection}-ctf-{source_phoneme}'")
            #     source_next_phons = source_rows.next_phoneme.unique()

            for target_inflection in study_inflections:
                for target_orig_phoneme in vbz_nns_next_phonemes:
                    # if target_true:
                    #     target_rows = inflection_instances.query(f"inflection.str.startswith('{target_inflection}-')", engine="python")
                    # else:
                    #     target_rows = broad_cf_instances.query(f"inflection.str.startswith('{target_inflection}-ctf-')", engine="python")

                    expt_key = f"VBZNNS-src_{source_inflection}_{source_orig_phoneme}-tgt_{target_inflection}_{target_orig_phoneme}-{target_phoneme}"
                    expts[expt_key] = {
                        "base_query": f"base_inflection == '{source_inflection}' and counterfactual == False and orig_post_divergence == '{source_orig_phoneme}' and next_phoneme == '{target_phoneme}'",
                        "inflected_query": f"base_inflection == '{target_inflection}' and counterfactual == True and orig_post_divergence == '{target_orig_phoneme}' and next_phoneme == '{target_phoneme}'",
                        "equivalence_keys": ["inflected_phones", "inflected"],
                        "prediction_equivalence_keys": ["to_inflected_phones"],
                    }

  0%|          | 0/38 [00:00<?, ?it/s]

In [27]:
# aci_tmp = pd.concat(all_cross_instances, ignore_index=True)

In [28]:
# experiment = expts["VBZNNS-src_VBZ_Z-tgt_VBZ_Z-D"]
# for spec in analogy_pseudocausal.iter_equivalences(
#     experiment, aci_tmp,
#     agg_src, num_samples=50, flat_idx_lookup=flat_idx_lookup,
#     seed=53,
# ):
#     from pprint import pprint
#     pprint(spec)
#     break
#     print(spec["to_label"])

## Prepare other counterfactual prefix/suffix inflections

In [29]:
m = Morphemes()

from collections import defaultdict, Counter
import itertools
import json
with open("/home/jgauthier/.morphemes/db.json", "r") as f:
    db = json.load(f)

morphdb = defaultdict(list)
for x in db["WORDS"].values():
    morphdb[str(x["Word"]).lower()].append(x)

m = Morphemes()
def parse(w):
    output = {}
    morph_db_results = morphdb.get(str(w).lower(), None)
    if morph_db_results is not None and len(morph_db_results) > 0:
        morph_db_result = morph_db_results[0]
        output["status"] = "FOUND_IN_DATABASE"
        output["word"] = w
        output["morpheme_count"] = morph_db_result["Nmorph"]
        segmentation = morph_db_result["MorphoLexSegm"]
        fragments = m._Morphemes__parse_segmentation(segmentation)
        if fragments is not None:
            output["tree"] = fragments
    else:
        output["status"] = "NOT_FOUND"
        output["word"] = w
        #not found words, ie names of people/places should be counted as 1 morpheme
        output["morpheme_count"] = 1
    return output


def parse_flat(w):
    flat = []
    def inner(analysis):
        if "text" in analysis and "type" in analysis:
            flat.append((analysis["text"], analysis["type"]))
        if "children" in analysis:
            for child in analysis["children"]:
                inner(child)
        return flat

    morph = parse(w)
    if "tree" not in morph:
        return []

    for el in morph["tree"]:
        inner(el)
    return flat


morph_counts = Counter()

def update_counts(analysis):
    if analysis is None:
        # berp
        return
    if "text" in analysis and "type" in analysis:
        morph_counts[analysis["text"], analysis["type"]] += 1
    if "children" in analysis:
        for child in analysis["children"]:
            update_counts(child)

for w in tqdm(ss.labels):
    analysis = parse(w)
    if "tree" not in analysis:
        continue
    for el in analysis["tree"]:
        update_counts(el)

  0%|          | 0/32046 [00:00<?, ?it/s]

In [30]:
morph_counts

Counter({('ly', 'bound'): 1085,
         ('ion', 'bound'): 1039,
         ('er', 'bound'): 839,
         ('y', 'bound'): 673,
         ('ate', 'bound'): 631,
         ('al', 'bound'): 558,
         ('un', 'prefix'): 390,
         ('able', 'bound'): 342,
         ('im', 'prefix'): 341,
         ('ity', 'bound'): 341,
         ('re', 'prefix'): 332,
         ('in', 'prefix'): 308,
         ('ness', 'bound'): 298,
         ('ant', 'bound'): 287,
         ('ious', 'bound'): 271,
         ('ic', 'bound'): 263,
         ('co', 'prefix'): 259,
         ('a', 'prefix'): 232,
         ('dis', 'prefix'): 230,
         ('en', 'bound'): 212,
         ('ive', 'bound'): 211,
         ('ance', 'bound'): 210,
         ('ory', 'bound'): 201,
         ('ment', 'bound'): 196,
         ('ful', 'bound'): 188,
         ('or', 'bound'): 171,
         ('est', 'bound'): 155,
         ('less', 'bound'): 148,
         ('en', 'prefix'): 140,
         ('ist', 'bound'): 130,
         ('de', 'prefix'): 128,
        

In [53]:
# false friends = items with same phonological form (incl. stress) as the morpheme, but not derived from it
study_morphemes = {
    ("ly", "bound"): {
        "type": "suffix",
        "target_phones": ("L", "IY"),
        "false_friends": "only family melancholy curly holy jolly pearly ugly folly italy monopoly silly scaly panoply early lily anomaly oily burly bully prickly golly mealy sicily steely unholy belly henley medley ali alley avonlea bailey barley bentley billy bingley cecily charley collie copley crawley croxley deadly dolly dudley duly elderly emily farley flee framley galilee gathercole glee harley jelly lee milly morley orderly ossoli plea polly volley",
    },
    ("ion", "bound"): {
        "type": "suffix",
        "target_phones": ("SH", "AH", "N"),
        "false_friends": "caution condition cushion fashion friction function grecian magician mansion martian mission motion nation notion ocean passion pension physician plantation politician portion position precaution question russian section session situation station tension tradition vacation volition",
    },
    # too many "er" this is a separate study :O
    ("ate", "bound"): {
        "type": "suffix",
        "target_phones": ("EY", "T"),
        "false_friends": "fate state hate rate late mate plate slate estate kate gate date bate enervate grate inmate caliphate prostrate crate sate reinstate pate underrate magnate schoolmate irate innate ornate skate tate cognate housemate",
    },
    # al: very few false friends

    ("un", "prefix"): {
        "type": "prefix",
        "target_phones": ("AH", "N"),
        "false_friends": "under until uncle understand understood unto understanding undertake underneath undertaking undertaken uncle's underwater undergo undulating undergone underground underbrush undertone understands uncles undulated underlying underworld undergoing underwent unctuous undertaker undergrowth undersigned underestimate undermine understsandings underwear undervalue underlings underhanded undersized undergoes underlies undertakers underbody undulations undersides undermined underhand",
    },
    # im: too hard to separate false friends; skip
    ("re", "prefix"): {
        "type": "prefix",
        "target_phones": ("R", "IY"),
        # this is hard too, may be false positives
        "false_friends": "really reached read reason remained regard remain reach result relief realized reading reasons reality regarded retired region reward remains retreat relieved related results reader remaining responded reasonable repeat reaching revealed realize recent reserve reasoning regarding remote response recently responsible relate reserved responsibility repeating regretted regards relieve readers republic resentment retire regions retreated resources remainder refer resulted refrain resultant released release rewarded referred reaches reads real realities realization reap reasonably reasoned reeds regis relation relations relationship relationships religion religions religious remark remarked remarkable repast reply replied report reported reports resembled resembling resist resistance resisted resolve resolved resource respect respectable respecting resplendent responsibilities restrain restrained restrictions resumed retain retained retiring retort retorted retreating return returned returning returns revenge rita rita's wreathed",
    },
}

In [57]:
min_num_tokens = 5

for label, spec in study_morphemes.items():
    print(f"Processing {label} ({spec['type']})")

    if spec["type"] == "suffix":
        matches = cut_phonemic_forms[cut_phonemic_forms.str.endswith(" " + " ".join(spec["target_phones"]))]
    elif spec["type"] == "prefix":
        matches = cut_phonemic_forms[cut_phonemic_forms.str.startswith(" ".join(spec["target_phones"]) + " ")]
    else:
        raise ValueError(f"Unknown morpheme type: {spec['type']}")

    false_friend_mask = matches.index.get_level_values("label").isin(spec["false_friends"].split())
    false_friend_matches = matches[false_friend_mask]
    true_matches = matches[~false_friend_mask]

    # filter out matches with fewer than min_num_tokens tokens
    true_matches = true_matches.groupby("label").filter(lambda xs: len(xs) >= min_num_tokens)
    false_friend_matches = false_friend_matches.groupby("label").filter(lambda xs: len(xs) >= min_num_tokens)

    print(f"Number true types: {len(true_matches.index.get_level_values('label').unique())} false types: {len(false_friend_matches.index.get_level_values('label').unique())}")
    print(f"Number true instances: {len(true_matches)} false instances: {len(false_friend_matches)}")

Processing ('ly', 'bound') (suffix)
Number true types: 434 false types: 51
Number true instances: 9404 false instances: 1186
Processing ('ion', 'bound') (suffix)
Number true types: 237 false types: 33
Number true instances: 4517 false instances: 1087
Processing ('ate', 'bound') (suffix)
Number true types: 38 false types: 14
Number true instances: 700 false instances: 635
Processing ('un', 'prefix') (prefix)
Number true types: 107 false types: 18
Number true instances: 1468 false instances: 618
Processing ('re', 'prefix') (prefix)
Number true types: 56 false types: 108
Number true instances: 805 false instances: 2227


In [35]:
# study_morph = ("re", "prefix")
# candidates = [x for x in ss.labels if x.startswith(study_morph[0])]
# candidates = ss.label_counts.loc[candidates].sort_values(ascending=False).index.tolist()
# out = " ".join([x for x in candidates if study_morph not in parse_flat(x)])
# # print with text wrap
# import textwrap
# print(textwrap.fill(out, width=80))

In [58]:
def get_counterfactual_forms(base_phones, post_divergence):
    """
    Find phonemic forms which have this base with a *distinct* post-divergence
    and whose post-divergence does not overlap with the original.
    """
    matches = cut_phonemic_forms[cut_phonemic_forms.str.startswith(base_phones + " ")]
    matches = matches[~matches.str.endswith(" " + post_divergence)].to_frame()
    if matches.empty:
        return pd.DataFrame()

    matches["post_divergence"] = matches.description.str.replace(base_phones + " ", "")
    # drop matches which overlap with the given post divergence (e.g. L IY and L IY Z)
    matches = matches[~matches.post_divergence.apply(lambda x: post_divergence.startswith(x))]
    matches = matches[~matches.post_divergence.str.startswith(post_divergence)]
    matches["inflected_idx"] = matches.index.get_level_values("label").map(label_to_idx)

    return matches

In [59]:
# true matches: counterfactual forms in which the base does really have an attested morpheme
# e.g. "distinctive" is a true match for "distinct" "ly" because in the base, "ly" is a morpheme
true_matches_ctf = []
# counterfactual forms in which the base does not have an attested morpheme
# e.g. "anomalous" is a false match for "anomaly" "ly" because in the base, "ly" is not a morpheme
false_matches_ctf = []

for key, spec in study_morphemes.items():
    if spec["type"] != "suffix":
        # not yet supported
        continue
    name = "_".join(key)
    print(name)

    matches = cut_phonemic_forms[cut_phonemic_forms.str.endswith(" " + " ".join(spec["target_phones"]))]

    true_matches = matches[~matches.index.get_level_values("label").isin(spec["false_friends"].split())]
    false_matches = matches[matches.index.get_level_values("label").isin(spec["false_friends"].split())]

    # # DEV
    # true_matches = true_matches.sample(min(50, len(true_matches)))
    
    true_matches_ctf_i = []
    for base_phones in tqdm(true_matches.str.replace(" " + " ".join(spec["target_phones"]), "").unique()):
        true_matches_ctf_ij = get_counterfactual_forms(base_phones, " ".join(spec["target_phones"])).reset_index()
        true_matches_ctf_ij["base_phones"] = base_phones
        true_matches_ctf_ij = true_matches_ctf_ij.rename(
            columns={"label": "inflected", "instance_idx": "inflected_instance_idx",
                     "description": "inflected_phones"})
        true_matches_ctf_i.append(true_matches_ctf_ij)
    true_matches_ctf.append(
        pd.concat(true_matches_ctf_i, ignore_index=True).assign(
            inflection=name, orig_post_divergence=" ".join(spec["target_phones"])))

    false_matches_ctf_i = []
    for base_phones in false_matches.str.replace(" " + " ".join(spec["target_phones"]), "").unique():
        false_matches_ctf_ij = get_counterfactual_forms(base_phones, " ".join(spec["target_phones"])).reset_index()
        false_matches_ctf_ij["base_phones"] = base_phones
        false_matches_ctf_ij = false_matches_ctf_ij.rename(
            columns={"label": "inflected", "instance_idx": "inflected_instance_idx",
                     "description": "inflected_phones"})
        false_matches_ctf_i.append(false_matches_ctf_ij)
    false_matches_ctf.append(
        pd.concat(false_matches_ctf_i, ignore_index=True).assign(
            inflection=f"{name}_false", orig_post_divergence=" ".join(spec["target_phones"])))
    
    # break # DEV

ly_bound


  0%|          | 0/1400 [00:00<?, ?it/s]

/tmp/ipykernel_1337122/2363790796.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(true_matches_ctf_i, ignore_index=True).assign(


ion_bound


/tmp/ipykernel_1337122/2363790796.py:44: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(false_matches_ctf_i, ignore_index=True).assign(


  0%|          | 0/689 [00:00<?, ?it/s]

/tmp/ipykernel_1337122/2363790796.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(true_matches_ctf_i, ignore_index=True).assign(


ate_bound


  0%|          | 0/183 [00:00<?, ?it/s]

/tmp/ipykernel_1337122/2363790796.py:32: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(true_matches_ctf_i, ignore_index=True).assign(
/tmp/ipykernel_1337122/2363790796.py:44: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(false_matches_ctf_i, ignore_index=True).assign(


In [60]:
true_matches_ctf_df = pd.concat(true_matches_ctf, ignore_index=True)
false_matches_ctf_df = pd.concat(false_matches_ctf, ignore_index=True)

all_matches_ctf = pd.concat([true_matches_ctf_df, false_matches_ctf_df], ignore_index=True)

# Ignore very long post divergences. Complicates analysis and this is likely
# to introduce stress differences too
pre_length = len(all_matches_ctf)
all_matches_ctf = all_matches_ctf[all_matches_ctf.post_divergence.str.count(" ") < 3]
post_length = len(all_matches_ctf)
print(f"Filtered {pre_length - post_length} instances with long post divergences. "
      f"Remaining {len(all_matches_ctf)} instances.")

all_matches_ctf["inflected_idx"] = all_matches_ctf.inflected.map(label_to_idx)
all_matches_ctf["next_phoneme"] = all_matches_ctf.post_divergence.str.split(" ").str[0]
all_matches_ctf["next_phoneme_idx"] = all_matches_ctf.base_phones.str.count(" ") + 1

all_matches_ctf["next_phoneme_cohort"] = all_matches_ctf.groupby(["inflection", "base_phones"]) \
    .post_divergence.transform(get_next_phoneme_cohort)
all_matches_ctf["counterfactual"] = False

Filtered 274858 instances with long post divergences. Remaining 249916 instances.


In [61]:
cf_trials = []
cf_next_phonemes = set(all_matches_ctf.next_phoneme.unique())
for next_phoneme in tqdm(cf_next_phonemes):
    match = all_matches_ctf[all_matches_ctf.next_phoneme != next_phoneme]
    match = match[match.next_phoneme_cohort.str.contains(f"|{next_phoneme}|", regex=False)]
    match = match.copy()

    match["post_divergence"] = next_phoneme
    match["next_phoneme"] = next_phoneme
    match["base_inflection"] = match.inflection
    match["inflection"] = match.inflection + f"-ctf-{next_phoneme}"
    match["inflected_phones"] = match.base_phones + " " + next_phoneme
    match["counterfactual"] = True
    match = match.drop(columns=["next_phoneme_cohort"])

    cf_trials.append(match)

cf_trials = pd.concat(cf_trials, ignore_index=True)

  0%|          | 0/56 [00:00<?, ?it/s]

In [62]:
all_cross_instances.append(cf_trials)
all_cross_instances.append(all_matches_ctf)

In [63]:
for next_phon in tqdm(cf_next_phonemes):
    for infl, type in study_morphemes.keys():
        if type != "bound":
            # not yet supported
            continue

        key = f"{infl}_{type}"
        for source_true in [True, False]:
            source_infl = key if source_true else f"{key}_false"

            for target_true in [True, False]:
                target_infl = key if target_true else f"{key}_false"

                expt_key = f"{infl}-src_{source_true}-tgt_{target_true}-{next_phon}"

                expts[expt_key] = {
                    "base_query": f"inflection == '{source_infl}' and counterfactual == False and next_phoneme == '{next_phon}'",
                    "inflected_query": f"base_inflection == '{target_infl}' and counterfactual == True and next_phoneme == '{next_phon}'",
                    "equivalence_keys": ["inflected_phones", "inflected"],
                    "prediction_equivalence_keys": ["to_inflected_phones"],
                }

  0%|          | 0/56 [00:00<?, ?it/s]

In [64]:
# experiment = expts["ly-src_True-tgt_False-L"]
# for spec in analogy_pseudocausal.iter_equivalences(
#     experiment, aci_tmp,
#     agg_src, num_samples=50, flat_idx_lookup=flat_idx_lookup,
#     seed=55,
# ):
#     from pprint import pprint
#     pprint(spec)
#     break
#     print(spec["to_label"])

## Overall summary

In [65]:
all_cross_instances = pd.concat(all_cross_instances, ignore_index=True)

In [66]:
assert (~all_cross_instances.inflected.isna()).all()

In [67]:
assert (~all_cross_instances.next_phoneme_idx.isna()).all()
assert (~all_cross_instances.next_phoneme.isna()).all()

In [68]:
# Pre-execute the queries and eliminate expts with very few samples
expts_to_drop = []
expts_to_keep = []
for i, (key, spec) in enumerate(tqdm(expts.items())):
    source_rows = all_cross_instances.query(spec["base_query"])
    target_rows = all_cross_instances.query(spec["inflected_query"])
    if len(source_rows.inflected.unique()) < 10 or len(target_rows.inflected.unique()) < 10:
        # print(f"Eliminating {key} due to insufficient samples", len(source_rows), len(target_rows))
        expts_to_drop.append(key)
    else:
        expts_to_keep.append((len(source_rows), len(target_rows), key))

    if i % 100 == 0:
        print(f"Processed {i} experiments, {len(expts_to_drop)} ({len(expts_to_drop) / (i + 1) * 100:.2f}%) dropped, {len(expts_to_keep)} kept")

  0%|          | 0/2040 [00:00<?, ?it/s]

Processed 0 experiments, 0 (0.00%) dropped, 1 kept
Processed 100 experiments, 60 (59.41%) dropped, 41 kept
Processed 200 experiments, 139 (69.15%) dropped, 62 kept
Processed 300 experiments, 223 (74.09%) dropped, 78 kept
Processed 400 experiments, 305 (76.06%) dropped, 96 kept
Processed 500 experiments, 395 (78.84%) dropped, 106 kept
Processed 600 experiments, 480 (79.87%) dropped, 121 kept
Processed 700 experiments, 575 (82.03%) dropped, 126 kept
Processed 800 experiments, 664 (82.90%) dropped, 137 kept
Processed 900 experiments, 753 (83.57%) dropped, 148 kept
Processed 1000 experiments, 843 (84.22%) dropped, 158 kept
Processed 1100 experiments, 930 (84.47%) dropped, 171 kept
Processed 1200 experiments, 1015 (84.51%) dropped, 186 kept
Processed 1300 experiments, 1114 (85.63%) dropped, 187 kept
Processed 1400 experiments, 1192 (85.08%) dropped, 209 kept
Processed 1500 experiments, 1271 (84.68%) dropped, 230 kept
Processed 1600 experiments, 1321 (82.51%) dropped, 280 kept
Processed 1700

In [69]:
len(expts), len(expts) - len(expts_to_drop)

(2040, 470)

In [70]:
all_cross_instances.to_parquet(f"{outdir}/all_cross_instances.parquet")

In [73]:
ss.to_hdf5(f"{outdir}/state_space_spec.h5")

/scratch/jgauthier/transformers3/lib/python3.11/site-packages/tables/attributeset.py:322: DataTypeWarning: Unsupported type for attribute 'labels_are_repr' in node '/'. Offending HDF5 class: 8
  value = self._g_getattr(self._v_node, name)
/userdata/jgauthier/projects/ideal-word-representations/src/analysis/state_space.py:89: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed,key->block0_values] [items->Index(['description'], dtype='object')]

  self.cuts.to_hdf(path, key=cuts_key, mode="a")


In [74]:
import json
with open(f"{outdir}/experiments.json", "w") as f:
    json.dump({name: expt for name, expt in expts.items() if name not in expts_to_drop}, f, indent=2)

In [75]:
# Write dummy files to make the pipeline happy.
!touch {outdir}/inflection_results.parquet
!touch {outdir}/most_common_allomorphs.csv
!touch {outdir}/false_friends.csv